<a href="https://colab.research.google.com/gist/fellipessanha/9c7c99e61e2ab1826fb92f27523ab219/teste-descricao-imagem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from torch import float16

# default: Load the model on the available device(s)

model_name = "Qwen/Qwen3-VL-4B-Instruct"
model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_name, torch_dtype=float16, device_map="auto"
)

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/64.7k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

In [ ]:
processor = AutoProcessor.from_pretrained(
    model_name,
    min_pixels=256 * 28 * 28,
    max_pixels=1024 * 28 * 28,
)

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

In [ ]:
import logging

def query_image(image_uri: str, query: str) -> str:
  logging.getLogger().setLevel(logging.CRITICAL)
  messages = [
      {
          "role": "user",
          "content": [
              {
                  "type": "image",
                  "image": image_uri,
              },
              {"type": "text", "text": query},
          ],
      }
  ]

  # Preparation for inference
  inputs = processor.apply_chat_template(
      messages,
      tokenize=True,
      add_generation_prompt=True,
      return_dict=True,
      return_tensors="pt"
  )
  inputs = inputs.to(model.device)

  # Inference: Generation of the output
  generated_ids = model.generate(**inputs, max_new_tokens=128)
  generated_ids_trimmed = [
      out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
  ]
  output_text = processor.batch_decode(
      generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
  )
  return output_text[0]

In [ ]:
q1 = query_image(
    "https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Frenattour.com.br%2Fwp-content%2Fuploads%2F2024%2F02%2FEdificio-do-itau-Faria-lima-3500-Sao-Paulo.jpeg&f=1&nofb=1&ipt=0317e42a4b3cfacd5232330f6f135aee2ea5692ba6eb2e55235a3c5805f9fca8&ipo=images",
    "how are people dressed in this image?",
)
print(q1)

In this image, the people visible are dressed in casual or everyday attire, consistent with a warm, sunny day in an urban environment.

Here’s what can be observed:

- **The man walking on the sidewalk** near the building entrance is wearing:
  - An orange t-shirt
  - Dark shorts or pants
  - He appears to be barefoot or wearing light footwear, which suggests warm weather.

- **Other pedestrians** (further down the sidewalk, near the street):
  - One person is wearing a white top and dark pants.
  - Another person is wearing a dark top and light-colored pants or shorts.
 


In [ ]:
q2 = query_image(
    "https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Frenattour.com.br%2Fwp-content%2Fuploads%2F2024%2F02%2FEdificio-do-itau-Faria-lima-3500-Sao-Paulo.jpeg&f=1&nofb=1&ipt=0317e42a4b3cfacd5232330f6f135aee2ea5692ba6eb2e55235a3c5805f9fca8&ipo=images",
    "Describe this image. what are the 2 main elements it contains?",
)
print(q2)

This image depicts the **Faria Lima Financial Center**, a modern, glass-clad office building located in **São Paulo, Brazil**.

### Key Details:

- **Building**: The structure is a striking example of contemporary architecture, featuring a reflective, angular glass facade that mirrors the surrounding sky and trees. The name “FARIA LIMA FINANCIAL CENTER” is clearly visible on a stone pillar at the entrance, along with the address “0400”.

- **Location**: The building is situated on **Avenida Faria Lima**, one of São Paulo’s main commercial and financial thoroughfares. The address “


In [ ]:
q3 = query_image(
    "https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Frenattour.com.br%2Fwp-content%2Fuploads%2F2024%2F02%2FEdificio-do-itau-Faria-lima-3500-Sao-Paulo.jpeg&f=1&nofb=1&ipt=0317e42a4b3cfacd5232330f6f135aee2ea5692ba6eb2e55235a3c5805f9fca8&ipo=images",
    "where was this picture taken? answer in portuguese and mandarin.",
)
print(q3)

/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


**Português:**  
Esta foto foi tirada em São Paulo, Brasil, na região central da cidade, próximo ao edifício da Faria Lima Financial Center, localizado na Avenida Paulista ou em uma rua adjacente.

**中文：**  
这张照片拍摄于巴西圣保罗市，位于城市中心区域，靠近法里亚·利马金融中心大楼，位于保利斯塔大道或其相邻街道附近。


In [ ]:
processor.image_processor

Qwen2VLImageProcessor {
  "do_convert_rgb": true,
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.5,
    0.5,
    0.5
  ],
  "image_processor_type": "Qwen2VLImageProcessor",
  "image_std": [
    0.5,
    0.5,
    0.5
  ],
  "merge_size": 2,
  "patch_size": 16,
  "resample": 3,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "longest_edge": 802816,
    "shortest_edge": 200704
  },
  "temporal_patch_size": 2
}

[]()